In [0]:
# Method: setup
# Objective: load the Gold tables that will be tested
from pyspark.sql import functions as F

fact         = spark.table("workspace.gold.fact_transactions")
dim_date     = spark.table("workspace.gold.dim_date")
dim_category = spark.table("workspace.gold.dim_category")
dim_account  = spark.table("workspace.gold.dim_account")

In [0]:
# Method: helper function
# Objective: run one check and record PASS/FAIL (0 violations = PASS)
results = []

def run_check(name, violations):
    status = "PASS" if violations == 0 else "FAIL"
    results.append((name, violations, status))
    print(f"{status} | {name} | violations: {violations}")

In [0]:
# Method: data quality checks (uniqueness, not null, relationships)
# Objective: is the fact table clean and fully linked to its dimensions?

# 1) UNIQUE: no transaction_id may appear twice
dup = fact.groupBy("transaction_id").count().filter("count > 1").count()
run_check("fact: transaction_id is unique", dup)

# 2) NOT NULL: no key may be empty
nulls = fact.filter(
    F.col("transaction_id").isNull() | F.col("date_key").isNull()
    | F.col("account_key").isNull() | F.col("category_key").isNull()
).count()
run_check("fact: keys are not null", nulls)

# 3) RELATIONSHIPS: every fact key must exist in its dimension (no orphans)
run_check("fact: date_key exists in dim_date",
          fact.join(dim_date, "date_key", "left_anti").count())
run_check("fact: account_key exists in dim_account",
          fact.join(dim_account, "account_key", "left_anti").count())
run_check("fact: category_key exists in dim_category",
          fact.join(dim_category, "category_key", "left_anti").count())

In [0]:
# Method: profiling (distinct values and ranges)
# Objective: discover which values exist in status, currency and amount, to define the accepted values
fact.groupBy("status").count().show()
fact.groupBy("currency").count().show()
fact.agg(
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
    F.sum(F.when(F.col("amount").isNull(), 1).otherwise(0)).alias("null_amount")
).show()

In [0]:
# Method: data quality checks (accepted values and range)
# Objective: do status, currency and amount contain only valid values?

# BR-05: status must be one of the 3 known values (NULL counts as a violation too)
run_check("fact: status has only accepted values",
          fact.filter(F.col("status").isNull()
                      | ~F.col("status").isin("completed", "pending", "failed")).count())

# BR-06: the platform works with a single currency
run_check("fact: currency is BRL",
          fact.filter(F.col("currency").isNull()
                      | (F.col("currency") != "BRL")).count())

# BR-07: amount must be filled and greater than zero
run_check("fact: amount is not null and > 0",
          fact.filter(F.col("amount").isNull() | (F.col("amount") <= 0)).count())

In [0]:
# Method: reconciliation (totals must match across layers)
# Objective: does the Gold layer close with the Silver layer, and the marts with the fact?
silver = spark.table("workspace.silver.transactions")
mart_cat = spark.table("workspace.gold.mart_monthly_category")
mart_fail = spark.table("workspace.gold.mart_failure_rate_category")

# 1) same number of rows: silver -> fact
run_check("reconciliation: fact rows = silver rows",
          abs(fact.count() - silver.count()))

# 2) same total amount: silver -> fact
run_check("reconciliation: fact amount = silver amount",
          float(abs(fact.agg(F.sum("amount")).first()[0]
                    - silver.agg(F.sum("amount")).first()[0])))

# 3) BR-01: category mart total = fact total of completed transactions
completed_total = fact.filter(F.col("status") == "completed").agg(F.sum("amount")).first()[0]
run_check("reconciliation: mart_monthly_category = fact (completed)",
          float(abs(completed_total - mart_cat.agg(F.sum("total_amount")).first()[0])))

# 4) BR-03: failure mart totals = fact totals
tot = mart_fail.agg(F.sum("total_transactions"), F.sum("failed_transactions")).first()
run_check("reconciliation: mart_failure_rate transactions = fact rows",
          abs(tot[0] - fact.count()))
run_check("reconciliation: mart_failure_rate failed = fact failed",
          abs(tot[1] - fact.filter(F.col("status") == "failed").count()))

In [0]:
# Method: summary + assert
# Objective: show every result and STOP the notebook with an error if any check failed
import pandas as pd

summary = pd.DataFrame(results, columns=["check_name", "violations", "status"])
display(summary)

failed = [r[0] for r in results if r[2] == "FAIL"]
assert len(failed) == 0, f"{len(failed)} data quality check(s) failed: {failed}"
print(f"All {len(results)} checks passed")